# Differential gene expression analysis

To perform differential gene expression analysis we have several alternatives and modes:
- Single-cell level
- Pseudobulk level

The `dotools_py` package includes two functions to automatically test for DEA between two conditions
for all the celltypes we have defined in our object, as well a consensus function to run both approaches.

In [1]:
# Set up
import anndata as ad
import dotools_py as do

adata = ad.read_h5ad('/Users/david/Downloads/Data10x/adata.h5ad')
adata

2025-06-26 14:39:49,301 - Jupyter enviroment detected. Using "inline" backend


AnnData object with n_obs × n_vars = 2801 × 18517
    obs: 'batch', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'n_counts', 'doublet_class', 'doublet_score', 'leiden', 'cell_type', 'autoAnnot', 'celltypist_conf_score', 'annotation', 'annotation_recluster'
    var: 'mean', 'std', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'annotation_colors', 'annotation_recluster_colors', 'batch_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_CCA', 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'logcounts', 'scaled'
    obsp: 'connectivities', 'distances'

## DEA at the single-cell level

Among the test we can use we have: wilcoxon, t-test, logistic regression, t-test with overestimation of the variance and the MAST test.
The MAST test can be run using the `do.tl.run_mast`, the other test can be run with `do.tl.rank_genes_groups`. Alternatively, we can use `do.tl.rank_genes_condition`, to use any of the test and automatically test for all the cell-types

To reduce the computation time, we are going to only use the NK cells

In [3]:
nk = adata[adata.obs.annotation == 'NK'].copy()

df = do.tl.rank_genes_condition(nk,
                                groupby='condition',
                                subset_by='annotation',
                                reference='healthy',
                                groups=['disease'],
                                method='mast',
                                get_results=True
                                )

2025-06-26 14:32:21,170 - Running DGEs for NK.
2025-06-26 14:32:21,172 - Running MAST test in R.
2025-06-26 14:32:21,227 - Running test for disease


Reading AnnData in R
`fData` has no primerid.  I'll make something up.
`cData` has no wellKey.  I'll make something up.
Assuming data assay in position 1, with name et is log-transformed.
Running MAST Test

Done!
Combining coefficients and standard errors
Calculating log-fold changes
Calculating likelihood ratio tests
Refitting on reduced model...

Done!
Saving DGE Table


In [4]:
df.head(10)

,GeneName,pvals,log2fc,padj,pts_ref,pts_group,groups,annotation
0,A1BG,0.001228,2.602636,0.022698,0.041995,0.195652,disease,NK
1,A1BG-AS1,0.779995,1.540265,1.000000,0.002625,0.000000,disease,NK
2,A2M,0.637401,0.069724,1.000000,0.005249,0.000000,disease,NK
3,A2M-AS1,0.526081,-0.222878,1.000000,0.175853,0.130435,disease,NK
4,A4GALT,1.000000,3.050085,1.000000,0.000000,0.000000,disease,NK
5,AAAS,0.688836,-0.761130,1.000000,0.047244,0.021739,disease,NK
6,AACS,0.381421,-1.352886,0.973504,0.013123,0.000000,disease,NK
7,AAED1,0.044028,0.794737,0.313925,0.091864,0.065217,disease,NK
8,AAGAB,0.524845,0.875611,1.000000,0.065617,0.108696,disease,NK
9,AAK1,0.223586,-0.585253,0.758544,0.464567,0.347826,disease,NK


## DEA at the pseudobulk level

To perform differential gene expression using a pseudobulk approach we can use `do.tl.rank_genes_deseq2`, which test between two conditions for each celltype. In this case we need to generate pseudo-replicates since we only have one sample per condition.

In [8]:
df = do.tl.rank_genes_deseq2(adata,
                             ctrl_cond='healthy',
                             disease_cond='disease',
                             cluster_key='annotation',
                             batch_key='batch',
                             condition_key='condition',
                             design='~condition',
                             min_cells=50,
                             min_counts=10,
                             pseudobulk_approach='sum',
                             technical_replicates=2,
                             get_results=True
                             )

2025-06-26 14:47:46,465 - Generating Pseudo-bulk data


Pseudo-bulked clusters:   0%|          | 0/5 [00:00<?, ?it/s]



2025-06-26 14:47:46,527 - The samples ['batch2'] have < 50 in cluster Monocytes. Skipping cluster


2025-06-26 14:48:32,252 - Removed 7269 genes for having less than 10 total counts


Pseudo-bulked clusters:  40%|████      | 2/5 [00:45<01:08, 22.89s/it]



2025-06-26 14:48:32,313 - The samples ['batch2'] have < 50 in cluster NK. Skipping cluster


2025-06-26 14:49:06,784 - Removed 11751 genes for having less than 10 total counts


...


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 4.55 seconds.

Fitting dispersion trend curve...
... done in 0.12 seconds.

Fitting MAP dispersions...
... done in 2.68 seconds.

Fitting LFCs...
... done in 0.35 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.22 seconds.

Fitting size factors...
... done in 0.00 seconds.



Log2 fold change & Wald test p-value: condition disease vs healthy
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG      40.991749       -0.327147  0.256535 -1.275253  0.202220  0.342368
A1BG-AS1   4.011296       -0.900327  1.151836 -0.781645  0.434423  0.586715
A2M-AS1    4.788911       -1.276831  0.879683 -1.451468  0.146650  0.269481
AAAS      15.465204       -1.653207  0.532257 -3.106033  0.001896  0.007760
AACS       7.065496        0.549139  0.570851  0.961965  0.336067  0.492749
...             ...             ...       ...       ...       ...       ...
ZXDB       5.163944        1.379532  0.870239  1.585234  0.112913  0.220824
ZXDC      18.940039       -0.621203  0.488125 -1.272631  0.203149  0.343594
ZYG11B    18.821602       -0.313112  0.473796 -0.660860  0.508702  0.652142
ZYX       57.500546       -0.699522  0.227834 -3.070310  0.002138  0.008653
ZZEF1     23.576412       -0.146092  0.424839 -0.343877  0.730939  0.827079

[11248 rows x 6 colu

Fitting dispersions...
... done in 2.63 seconds.

Fitting dispersion trend curve...
... done in 0.06 seconds.

Fitting MAP dispersions...
... done in 1.19 seconds.

Fitting LFCs...


Log2 fold change & Wald test p-value: condition disease vs healthy
        baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG    6.737215       -0.029695  0.803176 -0.036972  0.970508  0.986048
A4GALT  2.732445        4.145012  2.552348  1.624000  0.104376       NaN
AAED1   9.147367        1.757493  0.825969  2.127796  0.033354  0.119545
AAMP    5.404597        0.193578  0.893846  0.216567  0.828546       NaN
AARS    2.080618        0.661657  1.505794  0.439407  0.660367       NaN
...          ...             ...       ...       ...       ...       ...
ZWINT   2.013741        3.704640  2.612508  1.418040  0.156179       NaN
ZXDC    2.912471       -0.357209  1.234685 -0.289312  0.772343       NaN
ZYG11B  2.716911        1.159679  1.370116  0.846409  0.397325       NaN
ZYX     7.155642       -0.550084  0.767378 -0.716835  0.473476  0.692572
ZZEF1   2.770099       -0.526244  1.248439 -0.421522  0.673374       NaN

[6766 rows x 6 columns]


... done in 0.31 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.14 seconds.



In [9]:
df.head(10)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,group
A1BG,40.991749,-0.327147,0.256535,-1.275253,2.022197e-01,3.423676e-01,T_cells
A1BG-AS1,4.011296,-0.900327,1.151836,-0.781645,4.344232e-01,5.867145e-01,T_cells
A2M-AS1,4.788911,-1.276831,0.879683,-1.451468,1.466495e-01,2.694808e-01,T_cells
AAAS,15.465204,-1.653207,0.532257,-3.106033,1.896156e-03,7.760403e-03,T_cells
AACS,7.065496,0.549139,0.570851,0.961965,3.360670e-01,4.927494e-01,T_cells
AAED1,48.669525,1.874270,0.226715,8.267074,1.372864e-16,2.338828e-15,T_cells
AAGAB,17.077093,0.109023,0.485553,0.224534,8.223420e-01,8.916081e-01,T_cells
AAK1,265.833008,-0.274357,0.129534,-2.118035,3.417209e-02,8.766950e-02,T_cells
AAMDC,9.597257,-0.653105,0.684285,-0.954434,3.398641e-01,4.964226e-01,T_cells
AAMP,48.323315,0.066339,0.297887,0.222698,8.237703e-01,8.922745e-01,T_cells


## DEA consensus

Additionally, the `do.tl.rank_genes_consensus` allow to perform both single-cell and pseudo-bulk DEA at generate a dataframe that summarises everything.

In [4]:
df = do.tl.rank_genes_consensus(adata,
                                ctrl_cond='healthy',
                                disease_cond='disease',
                                cluster_key='annotation',
                                batch_key='batch',
                                condition_key='condition',
                                design='~condition',
                                min_cells=50,
                                min_counts=10,
                                pseudobulk_approach='sum',
                                technical_replicates=2,
                                get_results=True,
                                test='wilcoxon'
                                )

2025-06-26 14:45:16,590 - Running wilcoxon
2025-06-26 14:45:16,764 - Running DGEs for B_cells.
2025-06-26 14:45:16,767 - Running wilcoxon test.
ranking genes
    finished (0:00:00)


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


2025-06-26 14:45:17,023 - Running DGEs for Monocytes.
2025-06-26 14:45:17,026 - Running wilcoxon test.
ranking genes
    finished (0:00:00)
2025-06-26 14:45:17,241 - Running DGEs for NK.
2025-06-26 14:45:17,244 - Running wilcoxon test.
ranking genes
    finished (0:00:00)
2025-06-26 14:45:17,637 - Running DGEs for T_cells.
2025-06-26 14:45:17,640 - Running wilcoxon test.
ranking genes
    finished (0:00:01)
2025-06-26 14:45:19,421 - Running DGEs for pDC.
2025-06-26 14:45:19,431 - Running wilcoxon test.
ranking genes
    finished (0:00:00)
2025-06-26 14:45:19,488 - Running DESeq2
2025-06-26 14:45:19,489 - Generating Pseudo-bulk data


Pseudo-bulked clusters:   0%|          | 0/5 [00:00<?, ?it/s]



2025-06-26 14:45:19,507 - The samples ['batch2'] have < 50 in cluster Monocytes. Skipping cluster


2025-06-26 14:46:16,504 - Removed 7269 genes for having less than 10 total counts


...


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 4.21 seconds.

Fitting dispersion trend curve...
... done in 0.10 seconds.

Fitting MAP dispersions...
... done in 2.62 seconds.

Fitting LFCs...
... done in 0.34 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.28 seconds.

Fitting size factors...
... done in 0.00 seconds.



Log2 fold change & Wald test p-value: condition disease vs healthy
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG      41.033222       -0.326754  0.255464 -1.279061  0.200876  0.340148
A1BG-AS1   3.973769       -0.907264  1.114808 -0.813830  0.415742  0.570320
A2M-AS1    4.805903       -1.272804  1.053935 -1.207668  0.227175  0.370205
AAAS      15.516742       -1.651321  0.624221 -2.645411  0.008159  0.027218
AACS       7.062926        0.548971  0.737152  0.744719  0.456442  0.606738
...             ...             ...       ...       ...       ...       ...
ZXDB       5.201843        1.379622  0.670363  2.058023  0.039588  0.098661
ZXDC      19.042503       -0.616248  0.495743 -1.243081  0.213838  0.355996
ZYG11B    18.934837       -0.308992  0.486647 -0.634940  0.525467  0.666388
ZYX       57.532219       -0.699114  0.227834 -3.068517  0.002151  0.008697
ZZEF1     23.585426       -0.146119  0.421518 -0.346650  0.728854  0.826437

[11248 rows x 6 colu

Fitting dispersions...
... done in 2.63 seconds.

Fitting dispersion trend curve...
... done in 0.05 seconds.

Fitting MAP dispersions...
... done in 1.24 seconds.

Fitting LFCs...


Log2 fold change & Wald test p-value: condition disease vs healthy
        baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG    6.596135       -0.017481  0.650826 -0.026860  0.978571  0.991605
A4GALT  2.705101        4.144795  2.552556  1.623782  0.104422       NaN
AAED1   9.089977        1.754904  0.677311  2.590987  0.009570  0.050111
AAMP    5.394867        0.208365  0.907179  0.229685  0.818337       NaN
AARS    2.125389        0.626836  1.481788  0.423027  0.672276       NaN
...          ...             ...       ...       ...       ...       ...
ZWINT   2.052955        3.738361  2.654005  1.408573  0.158961       NaN
ZXDC    2.958134       -0.371789  1.180582 -0.314920  0.752822       NaN
ZYG11B  2.680134        1.158812  1.395457  0.830417  0.406303       NaN
ZYX     6.997370       -0.543364  0.784218 -0.692874  0.488389  0.704442
ZZEF1   2.790904       -0.548051  1.230454 -0.445406  0.656027       NaN

[6766 rows x 6 columns]
2025-06-26 14:47:06,324 - Genera

... done in 0.29 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.16 seconds.



In [6]:
df.head(10)

,GeneName,wilcox_score,log2fc,pvals,padj,pts_group,pts_ref,group,annotation,log2fc_DESeq2,stat_DESeq2,pval_DESeq2,padj_DESeq2,sc_signicant,DESeq2_signicant,consensus_significant,MeanExpr_batch1,MeanExpr_batch2
0,RPS4Y1,10.417766,32.312714,2.057288e-25,9.523701e-22,0.812865,0.000000,disease,B_cells,8.297896,3.469855,5.207392e-04,5.261636e-03,Yes,Yes,Yes,0.000000,1.846016
1,CD83,10.117800,4.033925,4.606540e-24,1.421655e-20,0.865497,0.166667,disease,B_cells,3.965669,9.920076,3.404961e-23,5.504687e-21,Yes,Yes,Yes,0.549657,2.565062
2,JUND,9.986894,2.114214,1.739467e-23,4.601388e-20,0.959064,0.916667,disease,B_cells,2.161438,19.891928,4.780157e-88,9.273504e-85,Yes,Yes,Yes,2.782248,4.198936
3,FOS,9.884114,4.776987,4.878770e-23,1.129252e-19,0.883041,0.305556,disease,B_cells,4.721803,19.779640,4.458661e-87,5.766535e-84,Yes,Yes,Yes,0.981424,3.844560
4,HSP90AA1,9.636548,3.000897,5.604088e-22,1.153010e-18,0.929825,0.597222,disease,B_cells,3.107457,17.664898,7.815425e-70,7.580962e-67,Yes,Yes,Yes,1.463146,3.316828
5,CREM,9.293894,5.343356,1.487451e-20,2.754314e-17,0.748538,0.055556,disease,B_cells,5.369147,7.956155,1.774679e-15,1.530167e-13,Yes,Yes,Yes,0.192004,2.261108
6,DUSP2,8.601924,7.102722,7.839049e-18,1.036826e-14,0.660819,0.027778,disease,B_cells,7.037163,6.789317,1.126657e-11,6.071429e-10,Yes,Yes,Yes,0.074408,2.452486
7,RGS1,8.586860,7.340136,8.937884e-18,1.103352e-14,0.654971,0.027778,disease,B_cells,7.462295,6.864169,6.687961e-12,3.760766e-10,Yes,Yes,Yes,0.076572,2.631500
8,YPEL5,8.577455,2.700320,9.699565e-18,1.122543e-14,0.853801,0.347222,disease,B_cells,2.852628,11.008297,3.485292e-28,7.954665e-26,Yes,Yes,Yes,0.964321,2.446569
9,EIF1,8.334270,0.915142,7.797797e-17,8.493636e-14,0.988304,1.000000,disease,B_cells,1.029711,7.408093,1.281280e-13,8.721693e-12,Yes,Yes,Yes,3.039560,3.651153


In [10]:
adata

AnnData object with n_obs × n_vars = 2801 × 18517
    obs: 'batch', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'n_counts', 'doublet_class', 'doublet_score', 'leiden', 'cell_type', 'autoAnnot', 'celltypist_conf_score', 'annotation', 'annotation_recluster'
    var: 'mean', 'std', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'annotation_colors', 'annotation_recluster_colors', 'batch_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap', 'rank_genes_deseq2', 'rank_genes_condition', 'rank_genes_consensus'
    obsm: 'X_CCA', 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'logcounts', 'scaled'
    obsp: 'connectivities', 'distances'

As we can appreciate, the results of the DEA will be saved in the uns attributed

In [12]:
!pip list

Package                   Version      Editable project location
------------------------- ------------ ---------------------------------------------------
absl-py                   2.2.2
adjustText                1.3.0
aiobotocore               2.21.1
aiohappyeyeballs          2.6.1
aiohttp                   3.11.16
aioitertools              0.12.0
aiosignal                 1.3.2
anndata                   0.11.4
annotated-types           0.7.0
annoy                     1.17.3
anyio                     4.7.0
appnope                   0.1.2
archspec                  0.2.3
argon2-cffi               21.3.0
argon2-cffi-bindings      21.2.0
array_api_compat          1.11.2
asciitree                 0.3.3
asttokens                 3.0.0
async-lru                 2.0.4
async-timeout             5.0.1
attrs                     25.3.0
babel                     2.16.0
backports.tarfile         1.2.0
bbknn                     1.6.0
beautifulsoup4            4.12.3
bleach                    6.2.0
